# BDM2/P1 Darcy: conservative high-order local flux

The MHM skeleton restricts a BDM2 normal trace while retaining the full interior flux moments. Pressure is discontinuous P1. The flux normal is conforming across fine and macro interfaces; tangential flux and pressure can remain discontinuous. This follows the mixed-local framework of [Durán et al. (2019)](https://doi.org/10.1016/j.cma.2019.05.013); the following are original analytical verifications, not reproduced table values.


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from pymhm import TriangleMesh
from pymhm.darcy_mixed import solve_darcy_bdm

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists())


## Quadratic pressure and exact linear flux

For `p=x²+y²`, permeability `K=[[2,.3],[.3,1]]`, and `f=-6`, the exact flux is `q=-2Kx`. BDM2 contains this flux exactly; the discrete P1 pressure equals the cellwise L2 projection and therefore has a nonzero error.


In [ ]:
tensor = np.array([[2., .3], [.3, 1.]])
solution = solve_darcy_bdm(
    TriangleMesh.unit_square(2), permeability=tensor, source=-6,
    dirichlet=lambda x: np.sum(x*x, axis=1),
)
flux_error = solution.flux_l2_error(lambda x: -2*x @ tensor)
assert flux_error < 3e-12
assert solution.divergence_l2_error(-6.) < 3e-12
assert np.max(np.abs(solution.conservation_residuals())) < 3e-12
{"pressure_L2": solution.l2_error(lambda x: np.sum(x*x, axis=1)), "flux_L2": flux_error}


## Cosine field: enriching the trace separately

The archived case uses `p=cos(πx)cos(πy)`, identity permeability and the actual nonhomogeneous boundary trace. It fixes 32 diagonal macrotriangles and local refinement four while changing only the unsplit macro trace degree. A high-order local solver does not eliminate the error of an under-resolved constant normal trace.


In [ ]:
records = json.loads((ROOT / "examples/results/darcy-bdm.json").read_text())
records["trace_study"]


In [ ]:
display(Image(filename=str(ROOT / "docs/figures/darcy-bdm/cosine-fields.png")))
display(Image(filename=str(ROOT / "docs/figures/darcy-bdm/convergence-and-trace.png")))


## Equilibrium and independent assembly

`fine_equilibrium_residuals()` returns all three P1 moments of `div(q)-f` per fine cell. `fine_conservation_residuals()` returns their sum, and `normal_flux_residuals()` checks the normal moments against the signed skeleton. For nonpolynomial f, these zero moments do not imply zero pointwise divergence error.

A separate native test compares all fields with a global DOLFINx/UFL and Basix BDM2/P1 assembly on the identical fine mesh. The macro trace is fully enriched for that discrete-space equivalence. No external comparison driver is needed to run this notebook.


In [ ]:
{"max_P1_equilibrium_moment": max(float(np.max(np.abs(v))) for v in solution.fine_equilibrium_residuals()),
 "max_normal_moment_defect": max(float(np.max(np.abs(v))) for v in solution.normal_flux_residuals())}
